# Tracking, Dynamic Aperture, and Local Momentum Aperture

This notebook demonstrates the tracking-related workflows from `examples/example_features.py`: turn-by-turn tracking, 2D dynamic aperture searches, off-momentum aperture scans, 3D dynamic aperture polyhedra, fast Touschek tracking, and local momentum aperture calculations.

Some cells may take noticeably longer than the basic optics notebook. Reduce `n_turns`, grid sizes, or point counts when trying the workflow interactively.

## 1. Prepare Imports

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
project_root = next(
    p for p in (cwd, *cwd.parents)
    if (p / "simplestoragering").exists() and (p / "examples").exists()
)

sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / "examples"))

print(f"Project root: {project_root}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import simplestoragering as ssr
from example_lattice import generate_ring, generate_ring_6D

plt.rcParams["figure.figsize"] = (7, 4)

## 2. Generate the Ring

The examples in this notebook use the 4D tracking workflow from `example_features.py`.

In [ ]:
ring = generate_ring()
for ele in ring.elements:
    if isinstance(ele, ssr.Octupole):
        ele.k3 = 0
ring.linear_optics()

print(f"length = {ring.length:.6f} m")
print(f"nux = {ring.nux:.6f}, nuy = {ring.nuy:.6f}")

## 3. Dynamic Aperture from Radial Lines

`NLine` searches the dynamic aperture boundary along a set of radial lines in transverse phase space.

In [ ]:
da_lines = ssr.NLine(n_lines=5, xmax=0.02, ymax=0.01, n_points=10)
da_lines.search(ring, n_turns=100)

plt.plot(da_lines.aperture[:, 0] * 1000, da_lines.aperture[:, 1] * 1000, marker="o")
plt.xlabel("x [mm]")
plt.ylabel("y [mm]")
plt.grid(True)
plt.show()

## 4. Dynamic Aperture on an XY Grid

`XYGrid` scans a rectangular grid and marks whether each initial condition survives for the requested number of turns.

In [ ]:
da_xy = ssr.XYGrid(xmax=0.02, nx=20, ymax=0.01, ny=10, delta=0)
da_xy.search(ring, n_turns=100)

plt.scatter(
    da_xy.data[:, 0] * 1000,
    da_xy.data[:, 1] * 1000,
    c=da_xy.data[:, 2],
    cmap="binary",
    marker="s",
)
plt.xlabel("x [mm]")
plt.ylabel("y [mm]")
plt.grid(True)
plt.show()

## 5. Off-Momentum Aperture on an X-Delta Grid

`XDeltaGrid` scans horizontal amplitude and momentum offset.

In [ ]:
da_xdelta = ssr.XDeltaGrid(xmax=0.02, nx=20, delta_max=0.05, ndelta=5)
da_xdelta.search(ring, n_turns=100)

plt.scatter(
    da_xdelta.data[:, 1] * 100,
    da_xdelta.data[:, 0] * 1000,
    c=da_xdelta.data[:, 2],
    cmap="binary",
    marker="s",
)
plt.xlabel("delta [%]")
plt.ylabel("x [mm]")
plt.grid(True)
plt.show()

## 6. Turn-by-Turn Tracking at Mark Elements

When `record=True`, particle coordinates are recorded at `Mark` elements. The example lattice has two `ss` marks.

In [ ]:
particle = [1e-3, 0, 0, 0, 0, 0]
ssr.symplectic_track(particle=particle, lattice=ring, n_turns=100, record=True)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

data0 = ring.mark["ss"][0].data
axes[0].scatter(data0[:, 0], data0[:, 1], s=12)
axes[0].set_title(f"s = {ring.mark['ss'][0].s:.3f} m")
axes[0].set_xlabel("x [m]")
axes[0].set_ylabel("px")
axes[0].grid(True)

data1 = ring.mark["ss"][1].data
axes[1].scatter(data1[:, 0], data1[:, 1], s=12)
axes[1].set_title(f"s = {ring.mark['ss'][1].s:.3f} m")
axes[1].set_xlabel("x [m]")
axes[1].set_ylabel("px")
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 7. 3D Dynamic Aperture Polyhedron

`DynamicAperturePolyhedron` extends the dynamic aperture search to transverse phase space and momentum offset. This cell can be slower than the previous grid examples.

In [ ]:
ring = generate_ring_6D()
ring.linear_optics()

da_3d = ssr.DynamicAperturePolyhedron(
    0.02,
    10,
    20,
    1,
    0.1,
    delta_list_m=np.linspace(-0.04, -0.1, 7),
    delta_list_p=np.linspace(0.02, 0.07, 6),
)

da_3d.search(ring)

## 8. Fast Touschek Tracking

The polyhedron object can be reused to estimate local momentum acceptance for fast Touschek tracking.

In [ ]:
fast_ma = da_3d.fast_Touschek_tracking(ring)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(fast_ma[:, 0], fast_ma[:, 1] * 100, color="C0")
ax.plot(fast_ma[:, 0], fast_ma[:, 2] * 100, color="C0")
ax.set_xlabel("s [m]")
ax.set_ylabel("delta [%]")
ax.set_title("Fast Touschek tracking")
ax.grid(True)
ssr.plot_layout_in_ax(ring.elements, ax.twinx())
plt.show()

## 9. Local Momentum Aperture

`LocalMomentumAperture` searches the positive and negative momentum limits along the ring. The save step is optional and is commented out to avoid writing files by default.

In [ ]:
lma = ssr.LocalMomentumAperture(ring, ds=10.0)
lma.search(n_turns=100)

# Optional: save the tracked local momentum aperture data.
# lma.save(filename="example_4D_LMA_data.csv", header="LMA data tracked by SimpleStorageRing")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(lma.s, lma.max_delta * 100, color="C0")
ax.plot(lma.s, lma.min_delta * 100, color="C0")
ax.set_xlabel("s [m]")
ax.set_ylabel("delta [%]")
ax.grid(True)
ssr.plot_layout_in_ax(ring.elements, ax.twinx())
plt.show()

## Notes

- Increase `n_turns`, grid sizes, or the number of aperture points for production studies.
- Keep the smaller values in this notebook for interactive demonstrations.
- For reproducible benchmark studies, move long scans to scripts under a dedicated benchmark or validation directory.